# Difficulty-Aligned Trajectory Matching

MTT 有一件事似乎做的太简单了：我们从专家轨迹中采样起点 $\theta_t^*$ 并且以此加上 $M$ 步推到终点 $\theta_{t+M}^*$，这个采样是简单的均匀采样。作者没有揭示使用均匀采样的原因，实际上这也许遵从了一个思想，那就是所有阶段的专家轨迹应该被平等对待。

但是是否有可能，这个采样非常重要？这就是 DATM 的出发点。我们指出，这个采样方式不仅非常重要，还应该根据合成数据集大小来选择。假设我们的合成数据集被限制在极低的量级内，我们应该更多采样早期专家轨迹，因为早期专家轨迹中模型正在学习普遍的常见的图案，这指导合成数据集给出更加普遍的梯度；假设我们的合成数据集被放宽到更加高的量级，我们应该加入后期专家轨迹的采样，因为后期专家轨迹中模型正在钻研颗粒度更细致的图案，合成数据集应该适当给出一些更罕见的梯度。

我们详细说。原文是 https://arxiv.org/pdf/2310.05773 Towards Lossless Dataset Distillation via Difficulty-Aligned Trajectory Matching。

# 基本逻辑

首先我们简称 Image per Class 为 IPC。IPC 对于蒸馏数据集而言决定了其应该选择的分布方式，选择更普遍的分布还是更细致的分布。根据 IPC，我们可以调整蒸馏数据集对应的专家轨迹采样时间段。

下面这张图展示了这一思想。我们希望 IPC 极小的数据集主要展示 Easy Patterns，而 IPC 更大的数据集展示一些 Hard Patterns。右侧图表展示了 DATM 在 CIFAR-10 上使用这一思想后的 State-of-the-Art 水平。

<img src="./assets/DATM.png" width="900" height="300">

我们假设完整专家轨迹是
$$\tau^*
=
\{\theta_t^*\mid 0\leq t\leq h\}$$
其中 $h$ 是终点时刻。

我们设置两个边界。$T^{-}$ 是允许采样的最早时间点，$T^{+}$ 是允许采样的最晚时间点。换言之，采样仅仅是
$$T^{-}\leq t\leq T^{+}$$
我们写开完整专家轨迹
$$\tau^*
=
\{
\theta_0^*,\ldots,
\underbrace{
\theta_{T^-}^*,\ldots,\theta_{T^+}^*
}_{\text{Sample}},
\ldots,\theta_h^*
\}$$

因此还可以引出一个技巧，被称为 easy-to-hard curriculum。实际上核心思想就是让采样时间点上届 $T^{+}$ 一开始保持在较低值，随着迭代次数增加逐渐增大。这样可以保证数据集蒸馏初期学习 Easy Patterns 而后期学习 Hard Patterns。写开是
$$T_{\mathrm{cur}}
:
T_{\mathrm{initial}}
\longrightarrow
T^{+}$$

最后的，我们来讨论一下 Soft Label 相关内容。TESLA 做了一个简单的 Training-free Soft Label 初始化，但是 DATM 对这里做了更细致的处理。

首先，如果专家模型给出了错误的 Soft Label 怎么办？这实际上非常有可能，因为专家模型也不是完全正确的。一个错误的 Soft Label 会导致数据集被指引向错误的方向蒸馏，这是我们不愿意看到的。

解决办法很简单，我们尽量减少这种可能的发生。我们使用那些专家模型判断正确类别的数据作为合成数据集初始化。换言之，对每个真实样本 $x_i$，先计算专家给出 logits
$$L_i
=
f_{\theta^*}(x_i)$$
其中 $L_i\in\mathbb R^C$，$C$ 是类别总数。

然后 softmax 得到 Soft Label
$$q_i
=
\operatorname{softmax}(L_i)$$

现在我们挑选出专家判断正确的样本
$$\mathcal D_{\mathrm{sub}}
=
\left\{
(x_i,q_i)
\;\middle|\;
(x_i,y_i)\in\mathcal D_{\mathrm{real}},
\;
\arg\max_c L_{i,c}=y_i
\right\}$$

再从 $\mathcal{D}_{sub}$ 中挑选子集得到合成数据集初始化
$$\mathcal D_{\mathrm{syn}}
=
\{(\widetilde x_j,\widetilde q_j)\}_{j=1}^{m}$$
其中 $\widetilde x_j$ 是可学习的合成图像，$\widetilde q_j$ 是可学习的 Soft Label 并且 $m\ll n$。

请注意，Soft Label 也是可以学习的，这和 TESLA 固定标签思想不一样。这种学习来源于我们最终损失会向标签回传一个梯度。

更多的，DATM 甚至决定让学习率可学习。这里的学习率指的是
$$\widehat\theta_{t+i+1}
=
\widehat\theta_{t+i}
-
\alpha
\nabla_{\widehat\theta_{t+i}}
\ell
\left(
\widehat\theta_{t+i},
b_{t+i}
\right)$$
此处的 $\alpha$。

因此对于计算图
$$(\mathcal D_{\mathrm{syn}},\alpha)
\longrightarrow
\widehat\theta_{t+1}
\longrightarrow
\widehat\theta_{t+2}
\longrightarrow
\cdots
\longrightarrow
\widehat\theta_{t+N}
\longrightarrow
\mathcal L_{\mathrm{match}}$$
其回传三个梯度
$$\nabla_{\widetilde x_j}
\mathcal L_{\mathrm{match}},
\qquad
\nabla_{\widetilde q_j}
\mathcal L_{\mathrm{match}},
\qquad
\nabla_{\alpha}
\mathcal L_{\mathrm{match}}$$
我们可以验证一件事，这三个梯度的计算图都是局部的。这意味着可以那就是按照 TESLA 的思想进行大幅度的显存优化。

# 蒸馏算法

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1:} \text{ Pipeline of our method} \\
\hline
\textbf{Input: } \{\tau^*\}\text{: set of expert parameter trajectories. } N\text{: update times of the surrogate network in} \\
\quad \text{each inner optimization. } M\text{: update times between the start and target expert parameters.} \\
\quad T^-, T, T^+\text{: lower, current upper, final upper bound of the sample range of } t. \, \mathcal{D}_{\text{real}}\text{: original} \\
\quad \text{dataset. } I\text{: interval for expanding the sampling range.} \\
\text{Sample a model } f_{\theta^*} \text{ from } \{\tau^*\}. \\
\text{Construct } \mathcal{D}_{\text{sub}} = \left\{ (x_i, \text{softmax}(L_i)) \mid (x_i, y_i) \in \mathcal{D}_{\text{real}} \textbf{ and } \text{argmax}(L_i) == y_i \right\}, \text{ where} \\
\quad L_i = f_{\theta^*}(x_i). \\
\text{Randomly sample data from } \mathcal{D}_{\text{sub}} \text{ to initialize synthetic dataset } \mathcal{D}_{\text{syn}}. \\
\begin{aligned}
& \textbf{for } \text{iteration} \leftarrow 0 \textbf{ to } \text{max\_iteration} \textbf{ do} \\
& \quad \text{Randomly sample an expert training trajectory } \tau^* \in \{\tau^*\} \text{ with } \tau^* = \{\theta_i^*\}_0^n \\
& \quad \text{Select random start timestamp } t, \text{ where } T^- \le t \le T \\
& \quad \text{Sample } \theta_t^*, \, \theta_{t+M}^* \text{ from } \tau^*, \text{ initialize } \hat{\theta}_t = \theta_t^* \\
& \quad \textbf{for } i \leftarrow 0 \textbf{ to } N-1 \textbf{ do} \\
& \quad\quad b_{t+i} \sim \mathcal{D}_{\text{syn}} \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \, \triangleright \text{sample a mini-batch of distilled dataset} \\
& \quad\quad \hat{\theta}_{t+i+1} = \hat{\theta}_{t+i} - \alpha \nabla \ell(\hat{\theta}_{t+i}, b_{t+i}) \quad \,\, \triangleright \text{update surrogate model with CE loss} \\
& \quad \textbf{for end} \\
& \quad \text{Compute matching loss between } \hat{\theta}_{t+N} \text{ and } \theta_{t+M}^* \text{ with Eq. 1} \\
& \quad \text{Update } (x_i, \text{softmax}(L_i)) \in \{b\}_t^{t+N-1} \text{ and } \alpha \text{ with respect to the matching loss} \\
& \quad \textbf{if } (\text{iteration} \% I == 0) \textbf{ and } (T < T^+) \textbf{ then} \\
& \quad\quad T = T + 1 \\
& \textbf{for end}
\end{aligned} \\
\textbf{Output: } \text{distilled dataset } \mathcal{D}_{\text{syn}} \text{ and learning rate } \alpha \\
\hline
\end{array}
$$